# Dependencies (for venv)

In [1]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


# Generate the dataset

In [2]:
import random
import torch

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')   
# torch.set_default_device(device)

# print(device)

In [3]:
def generate_objects(N=3, block_size=0.4, initial_tower=True):
    # 1. Generate random order for the goals
    idx_goals = [i for i in range(0, N)]
    random.shuffle(idx_goals)

    # 2. Generate initial order for the blocks by inverting the order of goals
    idx_objs = idx_goals[::-1]

    # 3. Generate goals
    # https://www.generationrobots.com/media/panda-franka-emika-datasheet.pdf
    # according to the datasheet, the Franka Panda can reach in an area of 0.855 [m] around itself
    max_reach_dist = 0.7 # [m]

    # select a random (x,y) position for the goal.
    x_goal = random.random() * max_reach_dist
    y_goal = random.random() * max_reach_dist

    pos_goals = [(x_goal, y_goal, i*block_size) for i in range(0, N)]

    # Generate initial positions
    if initial_tower:
        x_obj = random.random() * max_reach_dist
        y_obj = random.random() * max_reach_dist

        pos_objects = [(x_obj, y_obj, i*block_size) for i in range(0, N)]
    else:
        pos_objects = [(random.random() * max_reach_dist, random.random() * max_reach_dist, i*block_size) for i in range(0, N)]

    goals = {}
    objects = {}
    for i in range(0, N):
        goals[idx_goals[i]] = pos_goals[i]
        objects[idx_objs[i]] = pos_objects[i]

    print(f"goals (in desired order of final stacking):\n{goals}")
    print(f"initial object positions (in order of stacking):\n{objects}")

    return objects, goals, idx_objs, idx_goals


In [4]:
# Now solve the stacking problem by taking the highest block in the objects tower (last in order)
# and putting it into the lowest available goal (first in order)

# build the state vector, ordered nodes with categorical, position and binary features, followed by ordered goals with the same features
def build_state(objects, goals):
    s = []
    N = len(objects)
    for n in range(0, N):
        # one hot encode of blocks (all nodes are either blocks or goals, all objects are goals)
        s.append(1)
        s.append(0)

        # one hot encode of whether block is in goal
        s.append(int(objects[n] == goals[n]))
        s.append(int(objects[n] != goals[n]))

        # position of block
        for o in objects[n]: s.append(o)

    for n in range(0, N):
        # one hot encode of goals (all nodes are either blocks or goals, all objects are goals)
        s.append(0)
        s.append(1)

        # one hot encode of whether block is in goal (goal if filled)
        s.append(int(objects[n] == goals[n]))
        s.append(int(objects[n] == goals[n]))

        # position of block
        for o in goals[n]: s.append(o)

    return s

def build_example(N=3, block_size=0.2, initial_tower=True):
    objects, goals, idx_objs, idx_goals = generate_objects(N, block_size, initial_tower)
    
    tau = []    
    tau.append(build_state(objects, goals))

    for n in range(0, N):
        # print(f"step #{n}")
        
        o = N-1-n

        # print(f"choosing object #{idx_objs[o]} with pos {objects[idx_objs[o]]}")
        # print(f"moving object #{idx_objs[o]} to goal{goals[idx_objs[o]]}")

        objects[idx_objs[o]] = goals[idx_objs[o]]
        tau.append([o, o])

        # print(f"Updated object positions with goals:")
        # print(f"goals:\n{goals}")
        # print(f"current objects pos:\n{objects}")
        # print("\n")

        tau.append(build_state(objects, goals))

    # TODO (?): add the empty action to a complete graph
    # print(tau)
    return tau

In [5]:
# Alternatevely
# at each step, find the highest block that is not yet in its goal
# and move it to the lowest available goal
# repeat until each blocks is in its goal

# Observation to graph

In [6]:
# Now put the demonstration in graph format, using pytorch geometric

import torch
import torch.nn.functional as F
import torch_geometric
from torch_geometric.utils.convert import to_networkx
import torch_geometric.transforms as T
import networkx as nx

def build_dataset(N=3, N_EXAMPLES=50):
    data_list = []
    for i in range(N_EXAMPLES):
        # print(f"Building example #{i}")
        tau = build_example(N)

        states = tau[0:-1:2] # exclude the complete graph
        actions = tau[1::2]

        # print(len(states), states)
        # print(len(states), actions)

        # build edges, only once
        # graphs are always fully connected, undirected, and with no edge features and no self loops
        edges = torch.tensor([(i,j) for i in range(N*2) for j in range(N*2) if i != j]).t().contiguous()
        transform = T.Cartesian(norm=False, cat=False)
        # build nodes and labels
        for n in range(0, N):
            s = states[n]
            a = actions[n]

            # torch_geometric.data.Data is a single graph
            # x is the input graph, as a matrix of dimension [node, node_features]
            # I have N*2 nodes (N blocks, N goals) and 5 features each
            x = torch.reshape(torch.tensor(s), (N*2, -1)) # N*2 considers both goals and blocks
            
            # additive measurement noise
            # z = torch.zeros(x.shape)
            # z[:, 2:] = torch.rand((z[:, 2:].shape))*0.005
            # x += z

            # y is divided into two vectors, one hot encoded, probabilities for the block and the goal
            # outputs are probabilities for each index to match dimensions with the mask
            y = torch.Tensor.float(F.one_hot(torch.tensor(a), num_classes=N))
            # y = (y[:, 0].reshape(-1), y[:, 1].reshape(-1))
            # print(y)
            y = torch.flatten(y)
            print(y)

            # y = torch.tensor(a)
            
            # keeping track of pos is useful to apply to transform and generate edge_attr later
            pos = x[:, 4:]  
            # x = x[:, :4]

            object_mask = x[:, 0] == 1 # for minibatch splitting during training
            # print(object_mask)
            # y = torch.tensor(a)
            G = torch_geometric.data.Data(x=x, edge_index=edges, pos=pos, y=y, object_mask=object_mask)
            # embed the distance between the objects as an edge feature (in edge_attr)
            # Transform.Cartesian derives it directly from G.pos
            G = transform(G)

            # save all the graphs into a single data_list for use as a DataLoader later
            data_list.append(G)

        # print(data_list)
    return data_list

/home/emamaker/university/Università/magistrale/anno-2/semestre-1/nn/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/emamaker/university/Università/magistrale/anno-2/semestre-1/nn/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


# Build torch_geometric dataset from examples

In [7]:
# https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html#frequently-asked-questions
# just create a DataLoader starting from the list of example Data
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

train_dataset = build_dataset(N=3, N_EXAMPLES=50)
# train_dataset = build_dataset(N=4, N_EXAMPLES=1) + build_dataset(N=3, N_EXAMPLES=1)
# print("Training dataset: ", train_dataset)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True)
train2_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, pin_memory=True)
val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1, shuffle=True, pin_memory=True)
# val_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10) + build_dataset(N=4, N_EXAMPLES=10), batch_size=1, shuffle=True)
test_loader = DataLoader(build_dataset(N=3, N_EXAMPLES=10), batch_size=1, pin_memory=True)


goals (in desired order of final stacking):
{1: (0.3026288258535019, 0.4429554242762828, 0.0), 0: (0.3026288258535019, 0.4429554242762828, 0.2), 2: (0.3026288258535019, 0.4429554242762828, 0.4)}
initial object positions (in order of stacking):
{2: (0.2879067160792927, 0.21579970058920042, 0.0), 0: (0.2879067160792927, 0.21579970058920042, 0.2), 1: (0.2879067160792927, 0.21579970058920042, 0.4)}
tensor([0., 0., 1., 0., 0., 1.])
tensor([0., 1., 0., 0., 1., 0.])
tensor([1., 0., 0., 1., 0., 0.])
goals (in desired order of final stacking):
{1: (0.31154756011320656, 0.30697928987338535, 0.0), 0: (0.31154756011320656, 0.30697928987338535, 0.2), 2: (0.31154756011320656, 0.30697928987338535, 0.4)}
initial object positions (in order of stacking):
{2: (0.5285629304547231, 0.6673812150077831, 0.0), 0: (0.5285629304547231, 0.6673812150077831, 0.2), 1: (0.5285629304547231, 0.6673812150077831, 0.4)}
tensor([0., 0., 1., 0., 0., 1.])
tensor([0., 1., 0., 0., 1., 0.])
tensor([1., 0., 0., 1., 0., 0.])
goa

# Build NN

In [8]:
# # https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html
# # https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
# # https://colab.research.google.com/github/Alireza-Akhavan/graph-neural-network/blob/main/07-Graph_Classification_pyg.ipynb#scrollTo=CN3sRVuaQ88l
# # https://khairulislam.github.io/GNN/pages/graph/Graph_Classification.html

# # regarding the use of position (and generalization to unseen positions): https://github.com/pyg-team/pytorch_geometric/issues/3326
# # regarding fully connected graphs and the use of dense layers: https://github.com/pyg-team/pytorch_geometric/issues/964

# from torch import nn
# from torch_geometric.nn import DenseSAGEConv, DenseGATConv, GATv2Conv
# from torch_geometric.utils import to_dense_adj, to_dense_batch, to_nested_tensor

# import lightning as L
# from lightning.pytorch.callbacks import Callback, EarlyStopping
# from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

# hidden_dim=8
# dropout_p=0.5

# # Linear projection from input node features to hidden dimension
# input_proj = nn.Linear(7, hidden_dim)

# # Standard PyTorch Transformer Encoder
# encoder_layer = nn.TransformerEncoderLayer(
#     d_model=hidden_dim,
#     nhead=1,
#     dim_feedforward=hidden_dim * 2,
#     dropout=dropout_p,
#     batch_first=True  # Inputs shape: [batch_size, max_seq_len, hidden_dim]
# )
# transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
# # Output classifier head
# classifier = nn.Linear(hidden_dim, 1)


# for i, batch in enumerate(train_loader):
#     print(batch)
#     data = batch

#     # batch: [Total_Nodes_In_Batch]
#     x, batch, is_object = data.x, data.batch, data.object_mask
#     print(x, x.shape)

#     # 1. Project node features
#     h = input_proj(x)  # [Total_Nodes, hidden_dim]
#     print(h, h.shape)

#     # 2. Convert PyG flat batch into dense padded tensor: [batch_size, max_N, hidden_dim]
#     # mask is True for valid nodes, False for padded positions
#     x_dense, mask = to_dense_batch(h, batch, fill_value=0.0)
#     print(x_dense, x_dense.shape)

#     # PyTorch Transformer src_key_padding_mask expects True for PADDED tokens to ignore them
#     key_padding_mask = ~mask  # Shape: [batch_size, max_N]
#     print(key_padding_mask)

#     # 3. Transformer Self-Attention across fully connected nodes
#     out_dense = transformer(x_dense, src_key_padding_mask=key_padding_mask)
#     print(out_dense, out_dense.shape)

#     out_flat = out_dense[mask]

#     # 4. Project transformer representations to class logits: [batch_size, max_N, C]
#     logits = classifier(out_flat)
#     print(logits, logits.shape)

#     # 5. Extract objects and goals back into flat masked representations
#     # Extract flat logits and valid batch indices for object/goal subsets
#     objects_logits = logits[data.object_mask]
#     objects_batch = batch[data.object_mask]

#     # print(objects_logits, objects_logits.shape)
#     # print(objects_batch, objects_batch.shape)
    
#     goals_logits = logits[~data.object_mask]
#     goals_batch = batch[~data.object_mask]
    
#     # since these outputs are interpreted as unnormalized logits, the fill value should be a large negative number to discourage the choice
#     objects_dense, objects_mask = to_dense_batch(objects_logits, objects_batch, fill_value=-1e3)
#     goals_dense, goals_mask = to_dense_batch(goals_logits, goals_batch, fill_value=-1e3)

#     print(objects_dense, goals_dense)

In [ ]:
# https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html
# https://pytorch-geometric.readthedocs.io/en/latest/get_started/introduction.html
# https://colab.research.google.com/github/Alireza-Akhavan/graph-neural-network/blob/main/07-Graph_Classification_pyg.ipynb#scrollTo=CN3sRVuaQ88l
# https://khairulislam.github.io/GNN/pages/graph/Graph_Classification.html

# regarding the use of position (and generalization to unseen positions): https://github.com/pyg-team/pytorch_geometric/issues/3326
# regarding fully connected graphs and the use of dense layers: https://github.com/pyg-team/pytorch_geometric/issues/964

from torch import nn
from torch_geometric.nn import DenseSAGEConv, DenseGATConv, GATv2Conv
from torch_geometric.utils import to_dense_adj, to_dense_batch, to_nested_tensor

import lightning as L
from lightning.pytorch.callbacks import Callback, EarlyStopping
from lightning.pytorch.callbacks.early_stopping import EarlyStoppingReason

# slopped
class GCN(torch.nn.Module):
    def __init__(self, hidden_dim=64, num_heads=4, num_layers=2, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(7, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, data):
        # x: [Total_Nodes_In_Batch, in_channels]
        # batch: [Total_Nodes_In_Batch]
        x, batch, is_object = data.x, data.batch, data.object_mask

        # 1. Project node features
        h = self.input_proj(x)  # [Total_Nodes, hidden_dim]

        # 2. Convert to dense padded tensor: [batch_size, max_N, hidden_dim]
        # mask is True for valid nodes, False for padding
        x_dense, mask = to_dense_batch(h, batch, fill_value=0.0)
        # print(x_dense, x_dense.shape)
        # print(mask, mask.shape)

        # 3. Transformer Self-Attention (key_padding_mask is True for PADDED tokens)
        key_padding_mask = ~mask
        out_dense = self.transformer(x_dense, src_key_padding_mask=key_padding_mask)
        # print(key_padding_mask, key_padding_mask.shape)
        # print(out_dense, out_dense.shape)

        out_flat = out_dense[mask]

        # 4. Project transformer representations to class logits: [batch_size, max_N, C]
        logits = self.classifier(out_flat)

        # 5. Extract objects and goals back into flat masked representations
        # Extract flat logits and valid batch indices for object/goal subsets
        objects_logits = logits[data.object_mask]
        objects_batch = batch[data.object_mask]

        # print(objects_logits, objects_logits.shape)
        # print(objects_batch, objects_batch.shape)
        
        goals_logits = logits[~data.object_mask]
        goals_batch = batch[~data.object_mask]
        
        # since these outputs are interpreted as unnormalized logits, the fill value should be a large negative number to discourage the choice
        objects_dense, objects_mask = to_dense_batch(objects_logits, objects_batch, fill_value=-1e3)
        goals_dense, goals_mask = to_dense_batch(goals_logits, goals_batch, fill_value=-1e3)

        return objects_dense, goals_dense, objects_mask, goals_mask

model = GCN(hidden_dim=32, dropout=0.0) #.to(device)
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-6)

for epoch in range(3000):
    model.train()
    Loss = torch.zeros((1))
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        objects_dense, goals_dense, objects_mask, goals_mask = model(batch)

        y_obj_dense, _ = to_dense_batch(batch.y[batch.object_mask], batch.batch[batch.object_mask], fill_value=-1)
        y_obj_dense = y_obj_dense.reshape(objects_dense.shape)

        y_goal_dense, _ = to_dense_batch(batch.y[~batch.object_mask], batch.batch[~batch.object_mask], fill_value=-1)
        y_goal_dense = y_goal_dense.reshape(goals_dense.shape)

        # print(objects_dense, objects_dense.shape, y_obj_dense, y_obj_dense.shape)
        
        l1 = F.cross_entropy(objects_dense, y_obj_dense, ignore_index=-1)
        loss = l1
        # l2 = F.cross_entropy(goals_dense, y_goal_dense, ignore_index=-1)
        # loss = l1+l2
        Loss += loss * batch.num_graphs

        loss.backward()
        optimizer.step()

    print(f'Training loss (CE): {Loss.item()}')

    model.eval()
    correct_obj = 0.
    correct_goal = 0.
    for i,batch in enumerate(train2_loader):
        objects_dense, goals_dense, objects_mask, goals_mask = model(batch)


        # print(objects_dense, objects_dense.shape)
        # print(batch.y[0])
        # print(torch.argmax(objects_dense)==batch.y[0], torch.argmax(objects_dense), batch.y[0])


        # take the argmax for the nodes in each batch
        correct_obj += (torch.argmax(objects_dense, dim=1) == batch.y[0]).sum().item()
        # correct_goal += (torch.argmax(goals_dense, dim=1) == batch.y[1]).sum().item()

        # print(correct_obj, correct_goal)

    acc_obj = correct_obj / i
    acc_goal = correct_goal / i
    print(f'Test Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')

    correct_obj = 0.
    correct_goal = 0.
    for i,batch in enumerate(val_loader):
        objects_dense, goals_dense, objects_mask, goals_mask = model(batch)
        correct_obj += (torch.argmax(objects_dense, dim=1) == batch.y[0]).sum().item()
        # correct_goal += (torch.argmax(goals_dense, dim=1) == batch.y[1]).sum().item()
    acc_obj = correct_obj / i
    acc_goal = correct_goal / i
    print(f'Validation Accuracy (objects/goals): {acc_obj:.4f}/{acc_goal:.4f}')

    # break

Training loss (CE): 166.1248016357422
tensor([[[1.7953],
         [1.6390],
         [1.6361]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(0.)
tensor(True) tensor(0) tensor(0.)
tensor([[[1.6331],
         [1.5138],
         [1.5079]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(0.)
tensor(True) tensor(0) tensor(0.)
tensor([[[1.5358],
         [1.5386],
         [1.5307]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(0.)
tensor(False) tensor(1) tensor(0.)
tensor([[[1.6505],
         [1.6519],
         [1.5090]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(1.)
tensor(True) tensor(1) tensor(1.)
tensor([[[1.5971],
         [1.6011],
         [1.5901]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(0.)
tensor(False) tensor(1) tensor(0.)
tensor([[[1.5283],
         [1.5218],
         [1.6511]]], grad_fn=<ViewBackward0>) torch.Size([1, 3, 1])
tensor(0.)
tensor(False) tensor(2) tensor(0.)
tensor([[[1.7966],
         [1.7956],
         [1.596

KeyboardInterrupt: 